<h1 style="text-align: center; font-size: 50px;"> Handwritten digit classification with keras MLflow integration </h1>

Notebook Overview
- Start Execution
- User Constants 
- Install and Import Libraries
- Configure Settings
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference


## Start Execution

In [1]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [3]:
start_time = time.time()  

logger.info("Notebook execution started.")

2025-08-29 20:30:17 - INFO - Notebook execution started.


## User Constants

In [4]:
DIGIT_BASE64 = "/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/wAALCAAcABwBAREA/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/9oACAEBAAA/APn+prW0uL66itbSCSe4lbbHFEpZmPoAOtaWt+FtZ8Ox28mqWghS4LCNlmSQblxuU7GO1huGQcHnpWPRXoOiWF/pfhiwh0K2ln8R+JfMWNoh89vaK2w7f7pdg2Wzwq9sk1X+IY03SItJ8JabMLk6QsjX1wpOJLuQr5gHsuxQP/rVw1Fen+EfFmueF/AN3qkup3C2yMbPR7QkBXmbJkk9SsYOcfd3MK8yd3lkaSRizsSzMxyST1JptFXLnVb280+ysJ5y9rYhxbx4AEe9tzdBzk9z7elU6K//2Q=="

## Install and Import Libraries

In [10]:
from PIL import Image
import base64
from io import BytesIO
import warnings                        
import logging  
import sys
import os

import numpy as np
import pandas as pd  

import matplotlib.pyplot as plt


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist

# ------------------------ MLflow Integration ------------------------
import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

sys.path.append("../src")
# Use the new models-from-code Logger
from mlflow.logger import Logger

# ------------------------ Utils Import ------------------------

from onnx_utils import ModelExportConfig
from utils import load_config

ModuleNotFoundError: No module named 'mlflow.logger'

## Configure Settings

In [ ]:
# ------------------------- MLflow Experiment Configuration -------------------------
EXPERIMENT_NAME = 'MNIST with TensorFlow'
RUN_NAME = "MNIST_Run"
MODEL_NAME = "MNIST_Model"
MODEL_PATH = "model_keras_mnist.keras"

In [ ]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuration paths
CONFIG_PATH = "../configs/config.yaml" 
DEMO_FOLDER = "../demo"

# Load configuration
config = load_config(CONFIG_PATH)

logger.info("✅ Configuration loaded successfully")

## Logging Model to MLflow

In [ ]:
# Use the new models-from-code Logger
from src.mlflow import Logger
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Prepare signature (same as before)
input_schema = Schema([ColSpec("string", name="digit")])
output_schema = Schema([ColSpec("long", name="prediction")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Save the trained keras model to disk so it can be included as an artifact
model.save(MODEL_PATH)


In [ ]:
# Load and preprocess MNIST data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0
y_cat_train = to_categorical(y_train, 10)
y_cat_test = to_categorical(y_test, 10)

# Build a simple CNN model
model = Sequential([
    Conv2D(32, kernel_size=(4, 4), activation='relu', input_shape=(28, 28, 1)),
    MaxPool2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_cat_train, epochs=4, validation_data=(x_test, y_cat_test))

In [ ]:
def base64_to_numpy(base64_string):
    """
    Convert base64 string to numpy array for MNIST digit prediction.
    """
    try:
        # Decode the base64 string
        image_data = base64.b64decode(base64_string)
        
        # Open the image using PIL
        image = Image.open(BytesIO(image_data))
        
        # Convert to grayscale if not already
        if image.mode != 'L':
            image = image.convert('L')
        
        # Resize to 28x28 if needed
        if image.size != (28, 28):
            image = image.resize((28, 28))
        
        # Convert to numpy array
        numpy_array = np.array(image)
        
        # Normalize pixel values to 0-1 range
        numpy_array = numpy_array.astype('float32') / 255.0
        
        # Reshape for model input (1, 28, 28, 1)
        numpy_array = numpy_array.reshape(1, 28, 28, 1)
        
        return numpy_array
        
    except Exception as e:
        logger.error(f"Error converting base64 to numpy: {str(e)}")
        raise

In [ ]:
logger.info(f'🚀 Starting the experiment: {EXPERIMENT_NAME}')

mlflow.set_tracking_uri('/phoenix/mlflow')
# Set the MLflow experiment name
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

# Start an MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:
    # Print the artifact URI for reference
    logger.info(f"📁 Run's Artifact URI: {run.info.artifact_uri}")
    
    # Evaluate model
    test_loss, test_accuracy = model.evaluate(x_test, y_cat_test, verbose=0)
    mlflow.log_metrics({"test_accuracy": test_accuracy})
    mlflow.log_metrics({"test_loss": test_loss})
    
    logger.info(f"📊 Test accuracy: {test_accuracy:.4f}")
    
    # Prepare model signature
    input_schema = Schema([ColSpec("string", name="digit")])
    output_schema = Schema([ColSpec("long", name="prediction")])
    signature = ModelSignature(inputs=input_schema, outputs=output_schema)
    
    # Ensure model file is saved for artifact inclusion
    if not os.path.exists(MODEL_PATH):
        model.save(MODEL_PATH)
    
    # Log the model to MLflow using the new Logger (models-from-code)
    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path=CONFIG_PATH,
        docs_path="../data",
        model_path=MODEL_PATH,
        demo_folder=DEMO_FOLDER
    )
    
    # Register the logged model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
    
    logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

## Fetching the Latest Model Version from MLflow

In [ ]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the model
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version  # Extract the latest model version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_model_version}")

# Print the latest model version and its signature
logger.info(f"Latest Model Version: {latest_model_version}")
logger.info(f"Model Signature: {model_info.signature}")

## Loading the Model and Running Inference

In [ ]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_model_version}")

# Base64 example
base = DIGIT_BASE64
numpy_image = base64_to_numpy(base)
# Image of the base64 example
plt.imshow(numpy_image.squeeze(), cmap= 'gray') 

base_input = pd.DataFrame({"digit": [base]})
# Prediction of base64
predictions = model.predict(base_input)

logger.info(f"Predicted class: {predictions}")

In [ ]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**Z by HP AI Studio**](https://zdocs.datascience.hp.com/docs/aistudio/overview).

In [ ]:
from src.mlflow import Logger
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Prepare signature and save model if not saved
input_schema = Schema([ColSpec("string", name="digit")])
output_schema = Schema([ColSpec("long", name="prediction")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)
if not os.path.exists(MODEL_PATH):
    model.save(MODEL_PATH)

# Use Logger to log model (models-from-code)
Logger.log_model(
    signature=signature,
    artifact_path=MODEL_NAME,
    config_path=CONFIG_PATH,
    docs_path=DEMO_FOLDER,
    model_path=MODEL_PATH
)
print("Logged model using src.mlflow.Logger")